# Microduck: policy standard su terreno inclinato

Esegue `alpha_walking.onnx` (la policy di camminata ufficiale) su una **rampa a inclinazione costante** scelta da te, e salva un video.

Il terreno è `FlatRampTerrainCfg` di microduck_rl (`src/mjlab_microduck/tasks/slope_terrain.py`) con angolo minimo = massimo: un breve piano, una rampa lunga 10 m in discesa verso +x, un piano di arrivo. Il robot parte a metà rampa.

**Prima di iniziare:** *Runtime → Change runtime type → T4 GPU*. Poi esegui le celle in ordine.

## 1. Controllo della GPU

In [ ]:
!nvidia-smi

## 2. Installazione

Installa `uv`, clona microduck_rl e installa le dipendenze (versioni fissate in `uv.lock`). Il primo `uv sync` richiede alcuni minuti.

In [ ]:
import os

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
os.environ["UV_HTTP_TIMEOUT"] = "600"   # evita timeout sui pacchetti CUDA grandi

if not os.path.exists("/content/microduck_rl"):
    !git clone https://github.com/pollen-robotics/microduck_rl /content/microduck_rl
%cd /content/microduck_rl
!uv sync

In [ ]:
# Verifica che torch nell'ambiente del progetto veda la GPU (deve stampare True)
!uv run python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.version.cuda)"

## 3. Rendering senza schermo (EGL)

In [ ]:
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

## 4. Scaricare la policy standard

Se ricevi un errore 401/404 il repo è privato: esegui `!uv run huggingface-cli login` con un token che ha accesso, oppure carica il file a mano in `/content/microduck_rl/policies/`.

In [ ]:
!uv run python -c "from huggingface_hub import list_repo_files; print('\n'.join(list_repo_files('pollen-robotics/microduck-policies')))"

In [ ]:
# Se nella lista il file è in una sottocartella, scrivi qui il percorso completo
POLICY_FILE = "alpha_walking.onnx"

!uv run python -c "from huggingface_hub import hf_hub_download; print(hf_hub_download('pollen-robotics/microduck-policies', '{POLICY_FILE}', local_dir='policies'))"
POLICY_PATH = os.path.join("policies", POLICY_FILE)
print("policy:", POLICY_PATH, os.path.exists(POLICY_PATH))

## 5. Lo script di simulazione

Parametri da riga di comando:

| Parametro | Significato | Default |
|---|---|---|
| `--deg` | inclinazione della rampa in gradi (0 = piano) | 10 |
| `--yaw-deg` | orientamento iniziale: 0 = rivolto verso valle (discesa), 180 = verso monte (salita), 90 = di traverso | 0 |
| `--vx` | comando fisso di velocità in avanti in m/s; se omesso, comandi casuali come nel training | casuali |
| `--seconds` | durata della simulazione | 10 |
| `--pushes` | lascia le spinte della modalità play (ogni 0,5-1 s) | tolte |

Con `--vx` il robot non ruota (velocità angolare comandata 0), quindi resta più o meno nella direzione iniziale: è il modo per scegliere salita, discesa o traverso. Con `--vx 0` sta fermo in piedi sulla pendenza.

In [ ]:
%%writefile cammina_pendenza.py
import argparse, math, sys

import imageio
import numpy as np
import torch

sys.path.insert(0, "scripts")
import mjlab_microduck.tasks  # noqa: F401  registra i task
from mjlab.envs import ManagerBasedRlEnv
from mjlab.terrains.terrain_generator import TerrainGeneratorCfg
from pickup_datagen import OnnxMLP
from mjlab_microduck.tasks.microduck_velocity_env_cfg import make_microduck_velocity_env_cfg
from mjlab_microduck.tasks.slope_terrain import FlatRampTerrainCfg

p = argparse.ArgumentParser()
p.add_argument("policy")
p.add_argument("--deg", type=float, default=10.0, help="inclinazione della rampa (gradi)")
p.add_argument("--yaw-deg", type=float, default=0.0, help="0 = discesa, 180 = salita, 90 = traverso")
p.add_argument("--vx", type=float, default=None, help="comando fisso in avanti (m/s); default: casuali")
p.add_argument("--seconds", type=float, default=10.0)
p.add_argument("--pushes", action="store_true", help="lascia le spinte della modalità play")
p.add_argument("--out", default="cammina_pendenza.mp4")
args = p.parse_args()

dev = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", dev)

# rough=True: contatti del terreno ammorbiditi e più iterazioni del solver (servono con i terreni generati)
cfg = make_microduck_velocity_env_cfg(play=True, rough=True)
cfg.scene.num_envs = 1

# --- Terreno: piano 1 m -> rampa 10 m a inclinazione costante (scende verso +x) -> piano 3 m ---
cfg.scene.terrain.terrain_type = "generator"
cfg.scene.terrain.terrain_generator = TerrainGeneratorCfg(
    size=(14.0, 6.0),
    num_rows=1,
    num_cols=1,
    curriculum=False,
    add_lights=False,
    sub_terrains={
        "rampa": FlatRampTerrainCfg(
            flat_length=1.0,
            ramp_length_range=(10.0, 10.0),
            runout_length=3.0,
            spawn_on_ramp=5.0,          # il robot parte a metà rampa
            deg_min=args.deg,
            deg_max=args.deg,           # min = max -> inclinazione costante
        )
    },
)
cfg.curriculum.pop("terrain_levels", None)

# --- Orientamento iniziale fisso ---
yaw = math.radians(args.yaw_deg)
cfg.events["reset_base"].params["pose_range"]["yaw"] = (yaw, yaw)

# --- Spinte ---
if not args.pushes:
    cfg.events.pop("push_robot", None)

# --- Comando di velocità fisso (opzionale) ---
if args.vx is not None:
    c = cfg.commands["twist"]
    c.ranges.lin_vel_x = (args.vx, args.vx)
    c.ranges.lin_vel_y = (0.0, 0.0)
    c.ranges.ang_vel_z = (0.0, 0.0)
    c.rel_standing_envs = 0.0
    c.rel_turn_in_place_envs = 0.0
    cfg.curriculum.pop("standing_envs", None)   # altrimenti il curriculum reimposta rel_standing_envs

env = ManagerBasedRlEnv(cfg=cfg, device=dev, render_mode="rgb_array")
policy = OnnxMLP(args.policy, dev)
robot = env.scene["robot"]


def frame():
    f = env.render()
    if hasattr(f, "cpu"):
        f = f.cpu().numpy()
    f = np.asarray(f)
    return f[0] if f.ndim == 4 else f  # una immagine per copia -> la prima


print(f"pendenza {args.deg:.1f} gradi | orientamento {args.yaw_deg:.0f} gradi | vx {args.vx}")
obs, _ = env.reset()
assert obs["actor"].shape[-1] == 61, f"osservazioni {obs['actor'].shape}: la policy si aspetta 61"
frames, cadute = [], 0
for t in range(int(args.seconds / env.step_dt)):
    obs, _, term, trunc, _ = env.step(policy(obs["actor"]))
    frames.append(frame())
    if t % 50 == 0:
        vx, vy, wz = obs["actor"][0, 48:51].tolist()
        x, y, z = robot.data.root_link_pos_w[0].tolist()
        print(f"t={t * env.step_dt:5.1f}s  comando vx={vx:+.2f} vy={vy:+.2f} wz={wz:+.2f} | posizione x={x:+.2f} z={z:+.2f}")
    if term.any():
        cadute += 1
        print(f"caduto a t={t * env.step_dt:.1f}s (il robot riparte da metà rampa)")

imageio.mimsave(args.out, frames, fps=round(1 / env.step_dt))
print(f"cadute: {cadute} | video: {args.out} ({len(frames)} frame)")
env.close()

## 6. Una prova: scegli i parametri

La prima esecuzione compila i kernel di MuJoCo Warp (qualche minuto); le successive sono veloci finché il runtime non si riavvia.

In [ ]:
PENDENZA_GRADI = 10   # inclinazione della rampa
ORIENTAMENTO = 0      # 0 = discesa, 180 = salita, 90 = traverso
VX = 0.2              # m/s in avanti; scrivi None per comandi casuali
SECONDI = 10

vx_arg = "" if VX is None else f"--vx {VX}"
!uv run --with "imageio[ffmpeg]" python cammina_pendenza.py {POLICY_PATH} --deg {PENDENZA_GRADI} --yaw-deg {ORIENTAMENTO} {vx_arg} --seconds {SECONDI}

from IPython.display import Video
Video("cammina_pendenza.mp4", embed=True, width=640)

## 7. Confronto tra pendenze

Stessa direzione e stesso comando, inclinazione crescente: si vede dove la policy inizia a cadere. Modifica le liste a piacere.

In [ ]:
PENDENZE = [0, 5, 10, 15, 20]
ORIENTAMENTO = 0   # 0 = discesa, 180 = salita
VX = 0.2

for deg in PENDENZE:
    print(f"===== {deg} gradi =====")
    !uv run --with "imageio[ffmpeg]" python cammina_pendenza.py {POLICY_PATH} --deg {deg} --yaw-deg {ORIENTAMENTO} --vx {VX} --seconds 10 --out pendenza_{deg}.mp4 | grep -E "cadute|caduto"

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode

for deg in PENDENZE:
    with open(f"pendenza_{deg}.mp4", "rb") as f:
        data = b64encode(f.read()).decode()
    display(HTML(f'<p><b>{deg} gradi</b></p><video width=480 controls src="data:video/mp4;base64,{data}"></video>'))

## Note

- **Riferimento:** nel training su terreno irregolare le pendenze arrivano a circa 5,7 gradi (`slope_range` fino a 0,10 in `MICRODUCK_ROUGH_TERRAINS_CFG`); la policy ufficiale `alpha_walking` è stata addestrata su **piano**. Tutto ciò che è sopra pochi gradi è fuori distribuzione.
- **Spazio sulla rampa:** il robot parte a 5 m dall'inizio della rampa, con 5 m in discesa e 5 m in salita. A 0,2 m/s per 10 s percorre circa 2 m: c'è margine. Per simulazioni più lunghe aumenta `ramp_length_range` e `size` nello script (la somma piano + rampa + arrivo deve stare in `size[0]`).
- **Dopo una caduta** l'ambiente fa il reset e il robot riparte da metà rampa: nel video si vede un salto.

## Se qualcosa non va

| Errore | Causa probabile | Cosa fare |
|---|---|---|
| `CUDA: False` nella cella 2 | Runtime senza GPU | *Runtime → Change runtime type → T4 GPU*, poi rieseguire tutto |
| Errore EGL / `gladLoadGL` in `env.render()` | Rendering EGL non disponibile | Rieseguire la cella 3; in alternativa `!apt-get install -y libosmesa6` e `MUJOCO_GL=osmesa` |
| `AssertionError: flat+ramp_max+runout ... must fit` | Rampa più lunga della tessera | Aumentare `size[0]` |
| `ValueError` con un nome di operazione da `OnnxMLP` | Il grafo ONNX usa operazioni diverse da Sub/Div/Gemm/Elu | Estendere `OnnxMLP` (sezione 4.5 del piano) |
| `AttributeError` su `rel_turn_in_place_envs` o `ranges` | Nomi dei campi del comando diversi | Incollare l'errore: si sistema lo script |
| Il robot parte sotto il terreno o cade nel vuoto | Origine del terreno non allineata | Provare `--deg 0`: se succede anche lì, il problema è nella costruzione del terreno |